<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/vllm/vllm_tpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vLLM on a Colab TPU (v5e-1)

Runtime → Change runtime type → **v5e-1 TPU**.

**Read this first.** One v5e chip has **16 GB of HBM**. `Qwen/Qwen3.8-27B` is about 27.8B parameters, roughly **55 GB in bf16**, so it cannot fit.
vLLM's TPU backend loads safetensors weights (bf16 or supported quantizations), not the GGUF Q4 file used in the llama.cpp notebook. There is no official FP8/AWQ/GPTQ build of this model to shrink it either.
This notebook therefore checks whether the model fits, and if not, **falls back to a smaller Qwen model** so you still get a working vLLM-on-TPU run. Whether the 27B architecture is even supported by the TPU backend is unverified.

## Detect CPU, RAM, disk, and TPU env

In [ ]:
import os

!free -h | head -2
!df -h /content | tail -1
import os; print({k: v for k, v in os.environ.items() if "TPU" in k or "PJRT" in k})

# CPU + RAM every run (TPU VMs still have host CPU/RAM)
def _host():
    model, threads, sockets, core_ids = "unknown CPU", 0, set(), set()
    cur_phys = cur_core = None
    try:
        text = open("/proc/cpuinfo", encoding="utf-8", errors="replace").read()
    except OSError:
        text = ""
    for line in text.splitlines():
        if not line.strip():
            if cur_phys is not None and cur_core is not None:
                core_ids.add((cur_phys, cur_core))
            cur_phys = cur_core = None
            continue
        if ":" not in line:
            continue
        k, v = [x.strip() for x in line.split(":", 1)]
        if k == "model name" and model == "unknown CPU":
            model = v
        elif k == "processor":
            threads += 1
        elif k == "physical id":
            cur_phys = v; sockets.add(v)
        elif k == "core id":
            cur_core = v
    if cur_phys is not None and cur_core is not None:
        core_ids.add((cur_phys, cur_core))
    threads = threads or (os.cpu_count() or 1)
    cores = len(core_ids) or threads
    mem = {}
    try:
        for line in open("/proc/meminfo", encoding="utf-8", errors="replace"):
            if ":" not in line: continue
            k, rest = line.split(":", 1)
            parts = rest.split()
            if parts:
                try: mem[k] = int(parts[0])
                except ValueError: pass
    except OSError:
        pass
    total_kb = mem.get("MemTotal", 0)
    avail_kb = mem.get("MemAvailable", mem.get("MemFree", 0))
    to_gb = lambda kb: kb / 1_000_000
    print(f"CPU: {model} — {cores} cores / {threads} threads" + (f" ({len(sockets)} sockets)" if len(sockets) > 1 else ""))
    print(f"RAM: {to_gb(total_kb):.1f} GB total, {to_gb(avail_kb):.1f} GB available ({to_gb(total_kb-avail_kb):.1f} GB used)")
    return model, cores, threads, to_gb(total_kb), to_gb(avail_kb)

CPU_MODEL, CPU_CORES, CPU_THREADS, RAM_GB, RAM_AVAIL_GB = _host()


## 2. Config

In [ ]:
TARGET_MODEL    = "Qwen/Qwen3.8-27B"   # what you asked for
TARGET_PARAMS_B = 27.8                  # billions of parameters
FALLBACK_MODEL  = "Qwen/Qwen3-4B"      # ~8 GB in bf16; fits one v5e chip
FALLBACK_PARAMS_B = 4.0

HBM_GB          = 16
BYTES_PER_PARAM = 2          # bf16
KV_AND_OVERHEAD_GB = 3       # rough headroom for KV cache and activations
MAX_MODEL_LEN   = 16384     # agents (dsh) need a long context
FORCE_TARGET    = False      # True = try the 27B anyway (expect an out-of-memory failure)
REFRESH_WHEELS = False   # True = re-download the packages and re-save them on Drive
TOOL_PARSER     = "hermes"   # vLLM tool-call parser for agents; hermes suits Qwen3. Other Qwen3.x models may need a different one (see the vLLM docs)

## 3. Does it fit?

In [ ]:
def need_gb(params_b): return params_b * BYTES_PER_PARAM + KV_AND_OVERHEAD_GB

print(f"{TARGET_MODEL}: needs about {need_gb(TARGET_PARAMS_B):.0f} GB, chip has {HBM_GB} GB")
if need_gb(TARGET_PARAMS_B) <= HBM_GB or FORCE_TARGET:
    MODEL_ID = TARGET_MODEL
else:
    MODEL_ID = FALLBACK_MODEL
    print(f"Does not fit. Falling back to {FALLBACK_MODEL} (about {need_gb(FALLBACK_PARAMS_B):.0f} GB).")
print("Using:", MODEL_ID)

## 4. Keep the Hugging Face cache on Google Drive (models are downloaded once and reused)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
HF_CACHE = "/content/drive/MyDrive/llm_models/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE          # inherited by the vllm server started below
print("Hugging Face cache:", HF_CACHE)

## 5. Install vLLM with the TPU backend (packages are saved on Google Drive after the first install)

In [ ]:
import sys, os, glob, subprocess

WHEELS = f"/content/drive/MyDrive/llm_models/builds/vllm_tpu_py{sys.version_info.major}{sys.version_info.minor}"
PKGS = ["vllm-tpu"]
# Package names and versions for the TPU backend change often; see https://github.com/vllm-project/tpu-inference
# If "vllm-tpu" fails, try PKGS = ["tpu-inference", "vllm"]

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    open("/content/pip_install.log", "a").write(r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError("pip failed:\n" + (r.stdout + r.stderr)[-3000:])

open("/content/pip_install.log", "w").close()
cached = bool(glob.glob(f"{WHEELS}/*.whl")) and not REFRESH_WHEELS
if cached:
    print("Installing from the wheels saved on Drive...")
    try:
        pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
    except RuntimeError as e:
        print("Cached install failed, rebuilding the wheel cache:\n", str(e)[:500])
        cached = False
if not cached:
    print("Downloading packages and saving them to Drive (slow, one time)...")
    os.makedirs(WHEELS, exist_ok=True)
    pip("wheel", "-q", "-w", WHEELS, *PKGS)
    pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
print("Done. Wheels:", len(glob.glob(f"{WHEELS}/*.whl")))

In [ ]:
import importlib.metadata as md_
for pkg in ("vllm", "vllm-tpu", "tpu-inference", "jax", "jaxlib"):
    try: print(pkg, md_.version(pkg))
    except md_.PackageNotFoundError: print(pkg, "not installed")
import jax; print(jax.devices())

## 6. Start the server (the first start compiles with XLA and can take several minutes)

In [ ]:
import subprocess, time, requests, secrets

import os
KEY_FILE = "/content/drive/MyDrive/llm_models/hub/api_key"   # stable key on private Drive, shared with the hub notebooks
if os.path.exists(KEY_FILE) and len(open(KEY_FILE).read().strip()) >= 24:
    API_KEY = open(KEY_FILE).read().strip()
else:
    os.makedirs(os.path.dirname(KEY_FILE), exist_ok=True)
    API_KEY = secrets.token_urlsafe(32)
    open(KEY_FILE, "w").write(API_KEY + "\n")

server = subprocess.Popen(
    ["vllm", "serve", MODEL_ID,
     "--max-model-len", str(MAX_MODEL_LEN),
     "--tensor-parallel-size", "1",
     "--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER,
     "--host", "127.0.0.1", "--port", "8000", "--api-key", API_KEY],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(600):                      # up to ~20 min
    if server.poll() is not None:
        print("server exited:\n", open("/content/vllm.log").read()[-4000:]); break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/vllm.log").read()[-4000:])

## 7. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions", headers={"Authorization": f"Bearer {API_KEY}"}, json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": temperature,
    }, timeout=900)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print(chat("Explain what a TPU is in two sentences."))

## 8. Connect your machine over Tailscale

This joins the Colab VM to your Tailscale tailnet and publishes the server over HTTPS at `https://colab-vllm-tpu.<your-tailnet>.ts.net`, reachable only from devices in your tailnet. Run your harness (dsh or any OpenAI-compatible client) **on your own machine** in the same tailnet, pointed at the Base URL and API key printed below. Nothing harness-related runs in this notebook.

Create a **reusable + ephemeral** auth key at https://login.tailscale.com/admin/settings/keys. The first run asks for it and saves it to private Drive (`MyDrive/llm_models/hub/tailscale_authkey`); later sessions reuse it. The API key is also kept on Drive, and the hostname stays the same, so your harness settings keep working across sessions.

In [ ]:
# Uses the hub package's tunnel helpers (standard library only; nothing to install).
import os, sys, subprocess
HUB_DIR = "/content/hub_repo"
if not os.path.isdir(f"{HUB_DIR}/colab_llm_hub"):
    subprocess.run(f"rm -rf {HUB_DIR} && git clone -q --depth 1 https://github.com/Biegle-W/google_colab_llm {HUB_DIR}",
                   shell=True, check=True)
if HUB_DIR not in sys.path:
    sys.path.insert(0, HUB_DIR)
from colab_llm_hub import tunnel, provider, idle_guard

IDLE_TIMEOUT_MIN = 10  # @param {type:"integer"}
# release this runtime after this many idle minutes (no cell, no request, no CPU/GPU/network work; 0 = never)
idle_guard.start(IDLE_TIMEOUT_MIN)

DRIVE_DIR   = "/content/drive/MyDrive/llm_models"
PORT        = 8000
TS_HOSTNAME = "colab-vllm-tpu"   # this server's name on your tailnet
TUNNEL      = "tailscale"  # @param ["tailscale", "cloudflare"]
# tailscale : private to your tailnet, stable URL (auth key saved on private Drive)
# cloudflare: no account; public https://<words>.trycloudflare.com URL, new every session (API key still applies)
TS_FUNNEL   = False   # True = also reachable from the public internet (the API key still applies)

if TUNNEL == "cloudflare":
    ts_proc, BASE = tunnel.start_cloudflare_quick(PORT, drive_dir=DRIVE_DIR, api_key=API_KEY)
else:
    ts_key = tunnel.load_or_ask_tailscale_authkey(f"{DRIVE_DIR}/hub/tailscale_authkey")
    ts_proc, BASE = tunnel.start_tailscale(PORT, ts_key, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
    del ts_key   # the key stays only in the private Drive file
MODEL_NAME = MODEL_ID
print("Base URL:", BASE + "/v1")
print("API key :", API_KEY)
print("Model   :", MODEL_NAME)
print()
print("On your machine" + (" (public URL, new each session)" if TUNNEL == "cloudflare" else " (same tailnet)")
      + ", any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print(f'curl {BASE}/v1/chat/completions -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" '
      f'-d \'{{"model": "{MODEL_NAME}", "messages": [{{"role": "user", "content": "Hello"}}]}}\'')

## Notes
- Check `/content/vllm.log` for errors. Unsupported architectures, out-of-memory, and version mismatches show up there.
- Host RAM on the Colab TPU runtime is limited (about 12 GB reported), which also constrains loading large checkpoints.
- Models are cached on Google Drive (`MyDrive/llm_models/hf_cache`), so later sessions reuse them. The first load from Drive can be slow.
- The API key (`MyDrive/llm_models/hub/api_key`) and Tailscale auth key are kept on private Drive and the Tailscale hostname is fixed, so the Base URL and key stay the same across sessions. If the previous session's node is still listed in your tailnet, Tailscale adds a suffix (`-1`); remove the old node to keep the name.
- The vLLM packages are saved on Drive (`MyDrive/llm_models/builds`) as wheels, so later sessions install without downloading. Set `REFRESH_WHEELS = True` to update them. Install errors are in `/content/pip_install.log`.